# A Machine Learning Model for Predicting Customer Churn Using XGBoost
#### Adam Hayward | September 2026

## Overview
___
This journal documents the process of developing a machine learning model for predicting customer churn for LoneStar Communications. The model is designed to support the company's Customer Retention Team by allowing customer information to be submitted to generate a prediction of the likelihood that the customer will churn. This journal details the data preparation, analysis, model development, evaluation, and interpretation used to create and assess the predictive model.

### Objective
___
The objective of this project is to develop a supervised machine learning classification model using XGBoost to predict customer churn to provide LoneStar Communications' Customer Retention Team with a data-driven assessment of a customer's churn risk. 

The model shall generate a churn probability and classify each customer into one of three risk levels: 
- ***Low Churn Risk*** (probability < 30%)

- ***Moderate Churn Risk*** (30%–59%)

-  ***High Churn Risk*** (≥ 60%)

 The model's performance will be evaluated using appropriate classification metrics to assess its effectiveness in predicting customer churn, including:
- **Accuracy** 

- **Precision**

- **Recall**

- **F1-Score**

- **ROC-AUC**

## Notebook Layout
---
This notebook is organized according to the **SEMMA** (*Sample, Explore, Modify, Model, and Assess*) methodology, in order to provide a structured framework for developing and evaluating the customer churn prediction model. The notebook begins with introductory sections covering the development environment, data source, and overall project setup before progressing through the five stages of the SEMMA methodology.

Each SEMMA stage is divided into relevant subsections that document the steps taken during model development. Custom functions used throughout the workflow are defined directly within the notebook, allowing the notebook to remain self-contained and reproducible. As a result, a downloaded copy of the notebook can be executed without requiring additional project-specific Python files; users only need to obtain the dataset identified in the Datasource section and install the required Python libraries.

The five SEMMA stages are organized as follows:

1. **Sample** – Load the dataset and establish the data used for model development.

2. **Explore** – Examine the dataset through descriptive statistics and exploratory data analysis to identify patterns, relationships, and potential issues.

3. **Modify** – Clean the data, engineer features, and prepare variables for use by the machine learning model.

4. **Model** – Develop and train the XGBoost classification model using the prepared dataset.

5. **Assess** – Evaluate model performance and interpret predictions using appropriate metrics and SHAP analysis.

Each step is presented sequentially, with each section building upon the previous one, allowing the complete model development process to be reproduced simply by executing the notebook's code from beginning to end.

## Developement Enviornment
___
The development environment consists of the programming language, libraries, and tools used throughout the analysis. These tools support data preparation, exploratory analysis, feature engineering, model development and evaluation, visualization, and interpretation of model results.

### Data Source
This project uses the Telco Customer Churn dataset, an openly available dataset published by IBM as part of its Customer Churn Prediction Code Pattern. The dataset contains customer account, service, and billing information used to analyze customer churn and develop the predictive model.


The dataset can be obtained, and downloaded as a CSV file from

#### Dataset Access
To simplify the workflow and make this notebook easier to run, the dataset will be loaded directly from its URL in IBM's GitHub repository. This eliminates the need to manually download and place the dataset in the notebook's working directory.

If preferred, the dataset can also be downloaded manually from the  **[IBM GitHub repository](https://github.com/IBM/telco-customer-churn-on-icp4d)** and loaded locally as a CSV file.

#### Dataset Description
IBM provides additional documentation describing the broader Telco Customer Churn sample, including explanations of many of the variables and the business context surrounding customer churn. The documentation is available through the **[IBM Community Telco Customer Churn article](https://community.ibm.com/community/user/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)**.

It is important to note that the IBM documentation describes a broader version of the Telco Customer Churn sample than the CSV used in this project. IBM's expanded sample It is important to note that the IBM documentation describes a broader version of the Telco Customer Churn sample than the dataset used in this project. IBM's expanded sample includes additional features that are not included in the dataset used here. However, the documentation provides useful background information and descriptions for many of the features that are included in this dataset.

### Python Libraries
The following Python libraries are used throughout the notebook to support data preparation, exploratory analysis, feature selection, model development, and evaluation.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Feature selection and model validation
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Machine learning model
from xgboost import XGBClassifier

# Notebook configuration
import warnings
warnings.filterwarnings("ignore")

# Project path configuration
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


##### Python Libraries Descriptions
- `Pandas` and `NumPy` are used for data manipulation and numerical operations.

- `Matplotlib` and `Seaborn` provide data visualization capabilities.

- `sklearn` (Scikit-learn) provides tools for feature selection, dataset partitioning, cross-validation, and model evaluation.

- `XGBoost` is used to develop the customer churn classification model. 

Additional Python modules are used to manage warnings and project file paths so that the notebook can locate project resources consistently. Function-specific libraries and modules are imported as needed within the functions where they are used.

## 1. Sample
---
During the Sample stage, the datasource used to develop the customer churn prediction model is obtained and loaded into a Pandas DataFrame. The DataFrame is used to examine and understand the dataset's structure, dimensions, feature types, and target variable. This initial inspection provides a baseline understanding of the data before proceeding to the Explore stage, where relationships and patterns within the dataset are analyzed in greater detail.

### Load and Inspect Dataset
The Telco Customer Churn dataset is loaded into a Pandas DataFrame from the application's `data` subdirectory. An initial inspection is performed to determine the dataset's dimensions and review its features and structure.

In [2]:
DATA_SOURCE_URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/"
    "data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(DATA_SOURCE_URL)

# Dataset dimensions.
display(df.shape)

# Available features.
df.columns.tolist()

(7043, 21)

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

### Feature Categorization
To support consistent analysis throughout the notebook, the dataset features are categorized based on their data type and analytical purpose. A metadata dictionary is created to identify numerical and categorical features, as well as the target variable. This metadata is used by custom functions throughout the analysis to automatically determine which features are appropriate for specific statistical analyses and visualizations. Centralizing this information also reduces repetitive code and helps maintain consistency as the dataset is explored and modified.

A separate copy of the dataframe is also created for use during the Explore stage. This working copy allows features to be temporarily recategorized, derived, or otherwise adjusted as needed for exploratory analysis without altering the original dataset. Any permanent data transformations or feature modifications will be performed later during the Modify stage. This separation helps preserve the integrity of the original data while maintaining a clear distinction between exploratory analysis and data preparation for modeling.

In [ ]:
column_metadata = {
    "customerID": {
        "category": "identifier",
        "unit": None,
        "analysis": None
    },
    "gender": {
        "category": "demographic",
        "unit": None,
        "analysis": "categorical"
    },
    "SeniorCitizen": {
        "category": "demographic",
        "unit": None,
        "analysis": "categorical"
    },
    "Partner": {
        "category": "demographic",
        "unit": None,
        "analysis": "categorical"
    },
    "Dependents": {
        "category": "demographic",
        "unit": None,
        "analysis": "categorical"
    },
    "tenure": {
        "category": "customer",
        "unit": "months",
        "analysis": "numeric"
    },
    "PhoneService": {
    "category": "service",
    "column": "PhoneService",
    "unit": None,
    "analysis": "categorical"
    },
    "MultipleLines": {
    "category": "service",
    "column": "MultipleLines",
    "unit": None,
    "analysis": "categorical"
    },
    "InternetService": {
        "category": "service",
        "column": "InternetService",
        "unit": None,
        "analysis": "categorical"
    },
    "OnlineSecurity": {
    "category": "service",
    "column": "OnlineSecurity",
    "unit": None,
    "analysis": "categorical"
    },
    "OnlineBackup": {
    "category": "service",
    "column": "OnlineBackup",
    "unit": None,
    "analysis": "categorical"
    },
    "DeviceProtection": {
    "category": "service",
    "column": "DeviceProtection",
    "unit": None,
    "analysis": "categorical"
    },
    "TechSupport": {
    "category": "service",
    "column": "TechSupport",
    "unit": None,
    "analysis": "categorical"
    },
    "StreamingTV": {
    "category": "service",
    "column": "StreamingTV",
    "unit": None,
    "analysis": "categorical"
    },
    "StreamingMovies": {
    "category": "service",
    "column": "StreamingMovies",
    "unit": None,
    "analysis": "categorical"
    },
    "Contract": {
        "category": "customer",
        "unit": None,
        "analysis": "categorical"
    },
    "PaperlessBilling": {
        "category": "billing",
        "unit": None,
        "analysis": "categorical"
    },
    "PaymentMethod": {
        "category": "billing",
        "unit": None,
        "analysis": "categorical"
    },
    "MonthlyCharges": {
        "category": "billing",
        "unit": "USD",
        "analysis": "numeric"
    },
    "TotalCharges": {
        "category": "billing",
        "type": "numeric",
        "unit": "USD",
        "analysis": "numeric"
    },
    "Churn": {
        "category": "target",
        "type": "",
        "unit": None,
        "analysis": "categorical"
    }
}

analysis_df = df.copy()

## 2. Explore
---
During the Explore stage, the dataset is further examined to identify any potential data quality issues patterns, feature distributions, and relationships between features that may influence the customer churn prediction model. Descriptive statistics and visualizations are used to understand the characteristics of individual features and investigate how customer attributes and services relate to churn.

### Data Quality Analysis & Remediation
 The dataset is evaluated for potential data quality issues that could affect the reliability of the analysis and model performance. This includes examining data types, missing values, duplicate records, inconsistent values, and other irregularities within the dataset. Identified issues are addressed using appropriate remediation techniques to ensure the data is suitable for subsequent analysis and model development.


#### Identify Missing & Blank Values
The dataset is examined for missing or blank values that could affect subsequent analysis. Both `Null` values and blank entries are identified to determine whether any features require remediation before proceeding with further analysis.

In [ ]:
missing_or_blank = analysis_df.isna() | analysis_df.astype(str).apply(lambda col: col.str.strip().eq(""))
missing_or_blank.sum()

#### Discovery
The report identified 11 missing or blank values under `TotalCharges`. 

The corresponding records are inspected to determine whether they represent a data entry issue or a meaningful characteristic of the customer records.

In [ ]:
display(analysis_df[missing_or_blank.any(axis=1)].head(11))

#### Discovery
Analysis of the affected records shows that each customer with a blank value for `TotalCharges` also has a `tenure` value of zero months. This indicates that these records represent customers who have not yet accumulated any charges. 

Because `TotalCharges` is an important numerical feature for the model and these records cannot provide a meaningful value for this variable, they will be excluded from the analysis.

In [ ]:
analysis_df = analysis_df[analysis_df["TotalCharges"].str.strip() != ""]

#### Identify Duplicate Records
The dataset is inspected for any duplicate records and determine whether there are any multiple rows that represent the same customer or observation. Identifying and addressing duplicate records helps prevent individual observations from being unintentionally overrepresented.

In [ ]:
print(f"Duplicate Recods: {analysis_df.duplicated().sum():,}")

#### Discovery
The report did not discover any duplicate records with in the dataset.

#### Review Feature Values & Consistency
Categorical features are examined for inconsistent or unexpected values that could affect analysis. The unique values within each categorical feature are reviewed to identify variations in spelling, capitalization, formatting, or category definitions that may need to be standardized.

In [ ]:
feature_summary = []

for feature in analysis_df.columns:
    unique_values = analysis_df[feature].unique()
    
    if len(unique_values) <= 4:
        values = ", ".join(map(str, unique_values))
    else:
        values = f"{analysis_df[feature].min()} to {analysis_df[feature].max()}"
    feature_summary.append({
        "Feature": feature,
        "Unique Values": len(unique_values),
        "Values / Range": values
    })

feature_summary_df = pd.DataFrame(feature_summary)

# Create figure
fig, ax = plt.subplots(figsize=(13, 9))
ax.axis("off")

# Create table
table_data = feature_summary_df.copy()

table = ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center",
    colWidths =[.2,.2,.6]
)

ax.set_title(
    "Feature Values & Consistency Review",
    fontsize=16,
    fontweight="bold",
    y=.98
)
for col in range(3):
    table[(0, col)].set_text_props(fontweight="bold", fontsize="12")

# Adjust table font size and scale.
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.subplots_adjust(
    top=0.90,
    right=0.90,
    hspace=0.35
    )
plt.show()

#### Discovery
- The `SeniorCitizen` feature is represented by binary values of `0` and `1`. To simplify analysis and maintain consistency with the other binary categorical features in the dataset, values of `0` will be converted to `No` and values of `1` will be converted to `Yes`.

- A new analysis feature will be created called `PartnerWithDependents` that will used to identify customers that have a `Partner` and `Dependens`.

- The `PaymentMethod` feature includes `(automatic)` within certain category names, which does not clearly distinguish the payment method itself from whether the payment is processed automatically.  Therefore, it will be removed from the category names and `autopay` will be created as a new analysis feature.

- `PhoneService` is a required for a customer to have `MultipleLines`.

- Customers subscribing to `InternetService` either have `DSL` or `Fiber optic`.

- Customer can only have `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, and `StreamingMovies` if subscribed to  `InternetService`.

#### Feature & MetaData Updates
Based on the feature assessment discoveries, features and metadata were updated to improve consistency, interpretability, and analytical usefulness.

In [ ]:
# Transform variable values.
analysis_df["SeniorCitizen"] = analysis_df["SeniorCitizen"].map({0:"No", 1: "Yes"})
analysis_df["PaymentMethod"] = analysis_df["PaymentMethod"].str.replace(" (automatic)", "", regex=False)

# Create new analysis Features.
analysis_df["PartnerWithDependents"] = np.where(
    (analysis_df["Partner"] == "Yes") &
    (analysis_df["Dependents"] == "Yes"),
    "Yes",
    "No"
)

analysis_df["AutoPay"] = np.where(
    analysis_df["PaymentMethod"].isin(["Bank transfer", "Credit card"]),
    "Yes",
    "No"
)

# Update feature metadata dictionary.
column_metadata["PhoneService"] = {
    "category": "service",
    "column": "PhoneService",
    "population": {
        "has_service": ["Yes"],
        "service_types": ["Yes"]
    },
    "sub_services": [
        "MultipleLines"
    ],
    "analysis_values": ["Yes", "No"],
    "unit": None,
    "analysis": "categorical"
}

column_metadata["InternetService"] = {
        "category": "service",
        "column": "InternetService",
        "population": {
            "has_service": ["DSL", "Fiber optic"],
            "service_types": ["DSL", "Fiber optic"]
        },
        "sub_services": [
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies"
        ],
        "analysis_values": ["Yes", "No"],
        "unit": None,
        "analysis": "categorical"
    }

column_metadata["PartnerWithDependents"] = {    
    "category": "demographic",
    "column": "PartnerWithDependents",
    "population": {
            "has_service": ["Yes"],
            },
    "type": "categorical",
    "unit": None,
    "analysis": "categorical"
}

column_metadata["AutoPay"] = {
    "category": "billing",
    "column": "AutoPay",
    "population": {
            "has_service": ["Yes"],
            },
    "type": "categorical",
    "unit": None,
    "analysis": "categorical"
}

for key in ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]:
    column_metadata.pop(key, None)

#### Feature Data Types
The data types of each feature are reviewed to ensure they appropriately represent the feature's values. Numerical features should be represented using numeric data types, while categorical features should use consistent categorical or string representations.

In [ ]:
type_summary = pd.DataFrame({
    "feature": analysis_df.columns,
    "dtype": analysis_df.dtypes.astype(str).values
})

fig, ax = plt.subplots(figsize=(7, 9))
ax.axis("off")

table_data = type_summary[["feature", "dtype"]].copy()

table_data.columns = ["Feature Name", "Data Type"]

table = ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center"
)

ax.set_title(
    "Feature Data Types",
    fontsize=16,
    fontweight="bold",
    y=.98
)
for col in range(2):
    table[(0, col)].set_text_props(fontweight="bold", fontsize="12")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.subplots_adjust(
    top=0.90,
    right=0.90,
    hspace=0.35
    )
plt.show()

#### Discovery
`TotalCharges` is stored as a `str` (string) even though it represents a numerical value. This data type is not appropriate for numerical analysis because the feature cannot be reliably treated as a continuous numerical variable in its current form. Therefore, it is converted to a numeric data type in `analysis_df`. 

In [ ]:
analysis_df["TotalCharges"] = pd.to_numeric(analysis_df['TotalCharges'], errors='coerce')

### Exploratory Data Analysis
Exploratory Data Analysis (EDA) is conducted to examine the distributions, relationships, and patterns within the dataset and to identify characteristics that may be associated with customer churn. Numerical and categorical features are analyzed using descriptive statistics and visualizations, with particular attention given to differences between customers who churned and those who remained. The findings from this analysis help identify potentially influential features and inform the feature engineering and modeling decisions made during the Modify and Model stages.


### Target Class Distribution
The distribution of the target variable, `Churn`, is examined to determine the proportion of customers who churned compared with those who remained.

In [ ]:
churn_summary = (
    analysis_df["Churn"]
    .value_counts()
    .rename_axis("Churn")
    .reset_index(name="Count")
)

churn_summary["Percentage"] = (
    churn_summary["Count"] / churn_summary["Count"].sum() * 100
)

churn_summary["Percentage"] = (
    churn_summary["Percentage"].map(lambda x: f"{x:.1f}%")
)

fig, ax = plt.subplots(figsize=(7, 2))
table_ax = ax
table_ax.axis("off")

table_data = churn_summary[["Churn","Count","Percentage"]].copy()

table_data.columns = ["Churn Status", "Customer Count", "Churn Rate"]

table = table_ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center"
)

table_ax.set_title(
    "Customer Churn Distribution",
    fontsize=16,
    fontweight="bold",
    y=.85
)
for col in range(3):
    table[(0, col)].set_text_props(fontweight="bold", fontsize="12")

for row in range(3):
    table[(row, 0)].set_text_props(fontweight="bold", fontsize="12")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(.8, 1.5)
plt.tight_layout()
plt.show()

### Analysis of the Target Class Distribution
The results show that the `Churn` variable is imbalanced, with a larger proportion of customers classified as No compared with Yes. This indicates that customers who remain with the company are more prevalent in the dataset than customers who churn. The class imbalance will be considered during model development and evaluation to ensure that model performance is not assessed using accuracy alone.

### Feature Distributions and Churn Rates
The distributions of individual features are examined to better understand the characteristics of the customer population and identify patterns associated with churn. Categorical features are compared by their observed churn rates, while numerical features are examined for their distributions and differences between customers who churned and those who remained.

### Demographic Features
Demographic features are examined to identify differences in churn behavior across demographic groups. The distribution of each demographic feature and its corresponding churn rate are analyzed to determine whether certain customer characteristics are associated with a higher or lower likelihood of churn.

##### Data Visualization Functions
The following functions creates a consolidated visualization of categorical features and their respective churn rates. Combining multiple features into a single figure provides a consistent view of the customer base and makes it easier to compare churn rates across groups. The function is designed to be reusable and will also be applied to other categorical feature groups throughout the exploratory analysis.

In [ ]:
import re
from matplotlib.gridspec import GridSpec

palette = {
    "Yes": "#0d6efd",
    "No": "#adb5bd",
    "Male": "#0d6efd",
    "Female": "#adb5bd",
    "Month-to-month": "#0d6efd",
    "One year": "#dc3545",
    "Two year": "#ffc107",
    "Q1": "#dc3545",
    "Q2": "#ffc107",
    "Q3": "#28a745",
    "Q4":"#007bff",
    "Electronic check": "#dc3545",
    "Mailed check":"#ffc107",
    "Bank transfer":"#28a745",
    "Credit card":"#007bff",
    "Internet Phone": "#dc3545",
    "Internet No Phone": "#ffc107",
    "No Internet Phone": "#28a745",
    "Internet Only": "#007bff",
    "DSL": "#0dcaf0",
    "Fiber optic": "#0d6efd"
}

churn_palette = {"No": "#0d6efd", "Yes": "#dc3545"}

def get_population(df, column):
    return df.loc[df[column].isin()].copy()

def format_column_name(column):
    column = re.sub(r'(?<!^)(?=[A-Z])', ' ', column)
    return column.title()

def pie_plot(data, x, ax, **plot_kwargs):
    counts = data[x].value_counts()
    label_map = plot_kwargs.pop("labels", None)
    
    if label_map is None:
        custom_labels = counts.index
    else:
        custom_labels = counts.index.map(label_map)
    colors = [palette[value] for value in counts.index]
        
    wedges, labels, autotexts = ax.pie(
        counts,
        autopct="%1.1f%%",
        labels=custom_labels,
        colors=colors,
        **plot_kwargs
    )

    for text in autotexts:
        text.set_color("white")
        text.set_fontweight("bold")

def add_bar_labels(ax, threshold=3):
    for container in ax.containers:
        labels = []
        for bar in container:
            height = bar.get_height()
            if height >= threshold:
                labels.append(f"{height:.0f}%")
            else:
                labels.append("")
        
        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=10,
            color='white',
            fontweight="bold"
        )

def get_churn_proportion(metadata, category, df):
    results = {}
    
    for column, info in metadata.items():
        
        if (info["analysis"] == "categorical" and info["category"] == category):
            
            if info.get("population_column"):
                population = get_population(df, column, info)
                analysis_df = population
            else:
                population = df
                analysis_df = df
                
            proportions = (
                pd.crosstab(
                    analysis_df[column],
                    analysis_df["Churn"],
                    normalize="index"
                ) * 100
            )
            
            results[column] = {
                "proportion": proportions,
                "population": analysis_df[column]
            }
            
    return results

def flatten_plot_features(churn_proportions):
    plot_features = []
    
    for column, feature in churn_proportions.items():
        # Add main feature
        plot_features.append(
            (column, feature)
        )
        # Add sub-services
        if "sub_services" in feature:
            for sub_service, sub_feature in feature["sub_services"].items():
                plot_features.append(
                    (sub_service, sub_feature)
                )
    return plot_features


def plot_feature_list(metadata, category, df,y_limits=None, **plot_kwargs):
    # Calculate churn proportions for each feature
    churn_proportions = get_churn_proportion(metadata, category, df)

    # Flatten the features into a list of column/feature pairs
    plot_features = flatten_plot_features(
        churn_proportions
    )

    row_count = len(plot_features)
    column_count = 2

    fig = plt.figure(figsize=(13, 5 * row_count))

    gs = GridSpec(
        row_count + 1,
        column_count,
        figure=fig,
        height_ratios=[0.25] + [1] * row_count
    )

    title_ax = fig.add_subplot(gs[0, :])
    title_ax.axis("off")
    title_ax.text(
        0.53,
        0.5,
        f"{category.title()} Features",
        ha="center",
        va="center",
        fontsize=22,
        fontweight="bold"
    )
    # Loop through each category to retrieve associated features
    for row, (column, feature) in enumerate(plot_features):
        population_ax = fig.add_subplot(gs[row + 1, 0])
        churn_ax = fig.add_subplot(gs[row + 1, 1])

        # Distribution Pie chart
        pie_plot(
            data=feature,
            x="population",
            ax=population_ax,
            **plot_kwargs
        )
            
        population_ax.set_title(
            f"{format_column_name(column)} Distribution",
            fontsize=14,
            fontweight="bold"
        )
        
        # Get churn proportion
        proportions = feature["proportion"]
        
        # Stacked bar chart
        proportions.plot(
            kind="bar",
            stacked=True,
            ax=churn_ax,
            rot=0,
            color=churn_palette
        )

        churn_ax.set_title(
            f"{format_column_name(column)} with Churn Proportion",
            fontsize=14,
            fontweight="bold"
        )

        churn_ax.set_xlabel("")
        churn_ax.set_ylabel("Percentage")
        add_bar_labels(churn_ax)
        churn_ax.legend(
            title="Churn",
            loc="upper left",
            bbox_to_anchor=(1, 1)
        )

        if y_limits:
            churn_ax.set_ylim(*y_limits)
        else:
            churn_ax.set_ylim(0, 100)

        for spine_name in ("top", "right"):
            churn_ax.spines[spine_name].set_visible(False)

    plt.subplots_adjust(
    top=0.90,
    right=0.90,
    hspace=0.35
    )
    plt.show()

#### Demographic Feature Visualization
The demographic features are visualized together to compare customer distributions and churn rates across demographic groups. Presenting the features within a single figure provides a consistent view of demographic patterns and makes differences in churn rates easier to identify.

In [ ]:
plot_feature_list(column_metadata, "demographic", analysis_df)

#### Demographic Analysis
- The similar churn rates across both `Male` and `Female` customers suggests that gender has limited association with customer churn within this dataset.

- Customers categorized as `SeniorCitizen` exhibit a higher churn rate than non-senior customers. 
- Customers categorized as not having a  `partner` display a higher churn rate compared with those that do. A similar pattern is observed for `Dependents`.

### Customer Features

The `tenure` and `Contract` features are examined to identify patterns associated with churn. `tenure` is analyzed to understand how the length of a customer's relationship with the company relates to churn, while `Contract` type is compared to determine whether the level of contractual commitment is associated with different churn rates. 

##### Data Visulization Functions
The following functions generate visualizations for numerical features by grouping their values into quartiles. These quartile groups are then able to be compared against categorical features to examine how the distribution of a numerical feature varies across categories and how these differences may relate to churn.

In [ ]:
from scipy.stats import norm

def distribution_by_quartile(data, metadata, column, ax):
    feature = data[column].dropna().copy()
    mean = feature.mean()
    std = feature.std()
    # Quartiles
    q1 = feature.quantile(0.25)
    q2 = feature.quantile(0.50)
    q3 = feature.quantile(0.75)

    # Plot actual customer population
    ax.hist(
        feature,
        bins=20,
        density=True,
        alpha=0.20,
        color="gray",
        edgecolor="black",
        label="Actual Population"
    )

    # Normal distribution
    values = np.linspace(
        max(0, feature.min()),
        feature.max(),
        1000
    )

    pdata = norm.pdf(values, mean, std)

    ax.plot(
        values,
        pdata,
        color="black",
        linewidth=2.5,
        label="Normal Distribution"
    )

    # Quartile labels
    q1_plus_1 = q1 + 1
    q2_plus_1 = q2 + 1
    q3_plus_1 = q3 + 1

    unit = metadata[column]["unit"]

    if unit == "USD":
        quartile_label = [
            f"Q1: 0–25%  (≤ ${q1:.0f})",
            f"Q2: 26–50%  (${q1_plus_1:.0f}–${q2:.0f})",
            f"Q3: 51–75%  (${q2_plus_1:.0f}–${q3:.0f})",
            f"Q4: 76–100%  (≥ ${q3_plus_1:.0f})"
        ]
    else:
        quartile_label = [
            f"Q1: 0–25%  (≤ {q1:.0f} {unit})",
            f"Q2: 26–50%  ({q1_plus_1:.0f}–{q2:.0f} {unit})",
            f"Q3: 51–75%  ({q2_plus_1:.0f}–{q3:.0f} {unit})",
            f"Q4: 76–100%  (≥ {q3_plus_1:.0f} {unit})"
        ]
    # Q1
    shade = (values >= feature.min()) & (values <= q1)
    ax.fill_between(
        values[shade],
        pdata[shade],
        color="#dc3545",
        alpha=0.40,
        label=quartile_label[0]
    )
    # Q2
    shade = (values >= q1) & (values <= q2)
    ax.fill_between(
        values[shade],
        pdata[shade],
        color="#ffc107",
        alpha=0.40,
        label=quartile_label[1]
    )
    # Q3
    shade = (values >= q2) & (values <= q3)
    ax.fill_between(
        values[shade],
        pdata[shade],
        color="#28a745",
        alpha=0.40,
        label=quartile_label[2]
    )
    # Q4
    shade = values >= q3
    ax.fill_between(
        values[shade],
        pdata[shade],
        color="#007bff",
        alpha=0.40,
        label=quartile_label[3]
    )
    # Quartile lines
    for q in [q1, q2, q3]:
        ax.axvline(
            q,
            color="black",
            linestyle="--",
            linewidth=1.5,
            alpha=0.7
        )
    ax.set_title(
        f"Customer {format_column_name(column)} Distribution by Quartile",
        fontsize=16,
        fontweight="bold"
    )
    
    for spine_name in ("top", "right"):
        ax.spines[spine_name].set_visible(False)
    ax.set_xlabel(f"{format_column_name(column)} {unit}")
    ax.set_ylabel("Density")
    ax.legend()


def feature_population_vs_churn(data, metadata, column, ax):
    data = data[[column, "Churn"]].dropna().copy()
    data["Churn"] = data["Churn"].map({
        "No": 0,
        "Yes": 1
    })

    feature = data[column]
    # Create quartiles
    data["quartile"] = pd.qcut(
        feature.rank(method="first"),
        q=4,
        labels=["Q1", "Q2", "Q3", "Q4"]
    )

    # Actual percentile values
    q1 = feature.quantile(0.25)
    q2 = feature.quantile(0.50)
    q3 = feature.quantile(0.75)

    # Churn rate by quartile
    churn_rates = (
        data.groupby(
            "quartile",
            observed=False
        )["Churn"]
        .mean() * 100
    )

    # Create histogram bins
    bin_edges = np.histogram_bin_edges(feature, bins="auto")
    bin_edges = np.unique(bin_edges)

    counts, bin_edges, patches = ax.hist(
        feature,
        bins=bin_edges,
        color="lightgray",
        edgecolor="white",
        alpha=0.8,
        label="Customer Population"
    )

    # Color histogram bins according to quartile
    for patch, left_edge in zip(patches, bin_edges[:-1]):
        if left_edge <= q1:
            patch.set_facecolor("#dc3545")
        elif left_edge <= q2:
            patch.set_facecolor("#ffc107")
        elif left_edge <= q3:
            patch.set_facecolor("#28a745")
        else:
            patch.set_facecolor("#007bff")

    # Normal distribution
    mean = feature.mean()
    std = feature.std()

    if std > 0:
        x = np.linspace(
            feature.min(),
            feature.max(),
            1000
        )

        pdf = norm.pdf(x, mean, std)

        bin_width = np.diff(bin_edges).mean()

        scaled_pdf = (pdf * len(feature) * bin_width)

        ax.plot(
            x,
            scaled_pdf,
            color="black",
            linewidth=2.5,
            label="Normal Distribution"
        )

    # Quartile lines
    for q in [q1, q2, q3]:
        ax.axvline(
            q,
            color="black",
            linestyle="--",
            linewidth=1.5,
            alpha=0.7
        )

    # Position labels within each quartile
    plot_min = feature.min()
    plot_max = feature.max()

    positions = [
        (plot_min + q1) / 2,
        (q1 + q2) / 2,
        (q2 + q3) / 2,
        (q3 + plot_max) / 2
    ]

    label_heights = {"Q1": 0.4, "Q2": 0.5, "Q3": 0.5, "Q4": 0.4}

    for position, quartile in zip(
        positions,
        ["Q1", "Q2", "Q3", "Q4"]
    ):

        rate = churn_rates.get(
            quartile,
            np.nan
        )

        ax.text(
            position,
            max(counts) * label_heights[quartile],
            f"{quartile}\n{rate:.1f}%\nchurn",
            ha="center",
            va="top",
            fontsize=12,
            fontweight="bold"
        )

    ax.set_title(
        f"{format_column_name(column)} Distribution & Churn Rate",
        fontsize=16,
        fontweight="bold"
    )

    unit = metadata[column]["unit"]

    ax.set_xlabel(f"{format_column_name(column)} ({unit})")
    ax.set_ylabel("Number of Customers")

    for spine_name in ("top", "right"):
        ax.spines[spine_name].set_visible(False)
    ax.legend()

def analyze_numerical_feature(data, metadata, column):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    distribution_by_quartile(data, metadata, column, axes[0])

    feature_population_vs_churn(data, metadata, column, axes[1])

    fig.suptitle(
        f"{format_column_name(column)} Analysis",
        fontsize=20,
        fontweight="bold",
        y=1.02
    )
    plt.show()

#### Numerical Customer Feature Visualization
`tenure` is visualized by dividing it into quartiles, allowing churn rates to be compared across different ranges of values. The visualizations also provide an overview of the overall distribution of the feature and help identify patterns or differences in churn behavior across the customer population.

In [ ]:
analyze_numerical_feature(analysis_df, column_metadata, "tenure")

#### Numerical Customer Feature Analysis
The visualizations show a notable relationship between `tenure` and `churn`. Although `Q1` is the shortest time frame, it has the highest churn rate of all other quartiles. Churn generally decreases as tenure increases, suggesting that customers with shorter relationships with the company are more likely to churn.

To further investigate this relationship, a new categorical feature, `TenureQuartile`, is created to represent customers according to their tenure quartile. This feature will allow tenure to be analyzed alongside other categorical customer features.

In [ ]:
analysis_df["TenureQuartile"] = pd.qcut(
    analysis_df["tenure"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

column_metadata["TenureQuartile"] = {
        "category": "customer",
        "type": "",
        "unit": None,
        "analysis": "categorical"
    }

#### Categorical Customer Feature Visualization
The categorical customer features are visualized to examine the distribution of `Contract` and the newly created `TenureQuartile` feature.

In [ ]:
plot_feature_list(column_metadata, "customer", analysis_df)

#### Categorical Customer Feature Analysis
- `Contract` appears to be a very strong indicator for customer churn. More than half of the customer population is enrolled in `Month-to-month` contract agreements, which also exhibits a comparatively high churn rate. This suggests that customers without long-term contractual commitments are more susceptible to churn and may represent a key target segment for retention efforts.

- When `tenure` is divided into quartiles, the customer population is relatively evenly distributed across the four groups. However, churn rates decline substantially with each successive quartile; with the churn rate decreasing by nearly half from one tenure group to the next. This indicates that newer customers are considerably more likely to churn, while customers who remain with the company longer become increasingly stable.

### Billing Features
Customer billing features are examined to identify patterns and potential relationships with churn. Both payment amounts and payment methods are analyzed to determine whether differences in billing behavior or payment preferences are associated with customer churn.

#### Numerical Billing Features Visualization
`MonthlyCharges` and `TotalCharges` are examined using quartiles to better understand their distributions and identify potential differences in churn across billing ranges. Grouping these continuous variables into quartiles allows churn rates to be compared across customers with relatively lower and higher billing amounts, making it easier to identify potential relationships between billing behavior and churn.

In [ ]:
analyze_numerical_feature(analysis_df, column_metadata, "MonthlyCharges")

analyze_numerical_feature(analysis_df, column_metadata, "TotalCharges")

#### Numerical Billing Feature Analysis
- Customers in the first `MonthlyCharges` quartile have a substantially lower churn rate compared with customers in the higher quartiles. Churn appears to peak among customers with monthly charges in the approximate range of $70–$110, suggesting that higher monthly billing amounts may be associated with an increased churn risk.

- Analysis of `TotalCharges` however shows a different pattern. Customers in the first quartile represent the smallest customer population, yet they have a substantially higher churn rate than customers in the remaining quartiles. Potentially reflecting the presence of newer customers that have churned who had less time to accumulate a higher amount for total charges.


`MonthlyChargesQuartile` is created as a new analysis feature so that monthly charges can be categorized into quartiles and compared with other categorical features.


`HighMonthlyCharges` and `HighTotalCharges` are also created as new features for analysis to identify customers whose charges fall within approximately the upper 25% of the observed value ranges.

In [ ]:
analysis_df["MonthlyChargesQuartile"] = pd.qcut(
    analysis_df["MonthlyCharges"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

analysis_df["HighMonthlyCharges"] = np.where(
    (analysis_df["MonthlyCharges"] >= 90),
    "Yes",
    "No"
)

analysis_df["HighTotalCharges"] = np.where(
    (analysis_df["TotalCharges"] >= 60),
    "Yes",
    "No"
)

column_metadata["MonthlyChargesQuartile"] = {
        "category": "billing",
        "column": "MonthlyChargesQuartile",
        "type": "",
        "unit": None,
        "analysis": "categorical"
    }

column_metadata["HighMonthlyCharges"] ={
            "category": "billing",
            "column": "HighMonthlyCharges",
            "population": {
                "has_service": ["Yes"],
            },
            "type": "",
            "unit": None,
            "analysis": "categorical"
}

column_metadata["HighTotalCharges"] ={
            "category": "billing",
            "column": "HighTotalCharges",
            "population": {
                "has_service": ["Yes"],
            },
            "type": "",
            "unit": None,
            "analysis": "categorical"
}


#### Categorical Billing Feature Visualization
Categorical billing features are examined to identify differences in churn across customer billing preferences and payment characteristics. The features visualized include `PaperlessBilling` and `PaymentMethod`, along with the derived `AutoPay`, `MonthlyChargesQuartile`, `HighMonthlyCharges`, and `HighTotalCharges`. 

In [ ]:
plot_feature_list(column_metadata, "billing", analysis_df)

#### Categorical Billing Feature Analysis
- Customers participating in `PaperlessBilling` represent a slight majority of the customer population; however, their churn rate is approximately twice that of customers who do not.

- `PaymentMethod` reveals that approximately one-third of the customer population uses `Electronic check`, making it the most common payment method and representing a slight majority compared with each of the other individual payment methods. However, customers using electronic checks have a churn rate of nearly 50%, while all other payment methods have relatively similar churn rates below 20%. This makes `Electronic check` a particularly strong categorical indicator of churn.

- `AutoPay` appears to have a similar customer population distribution to `PaperlessBilling`; however, the churn relationship is nearly reversed. This suggests that automatic payment enrollment may provide different information about churn than paperless billing alone, despite the two features having similar population distributions.

- `HighMonthlyCharges` represents roughly one-quarter of the customer population; however, its churn rate does not increase substantially compared with the rest of the customer population. One possible explanation is that higher monthly charges may reflect customers subscribing to multiple services.

- `HighTotalCharges` represents only about 6% of the customer population and has a churn rate similar to customers with lower monthly charges. A possible explanation is that this group contains customers with longer tenures and have accumulated higher total charges; potentially offsetting the increased churn risk that may otherwise be associated with higher billing amounts.

### Service Features 
Phone and internet subscription features are examined to understand the distribution of customers across the primary services and their associated service options and subservics. This visualization will aid in evaluating both the proportion of customers subscribing to each service and the extent to which customers subscribe to additional service features within each category.

#### Bundeld Phone & Internet Service Visualization
`PhoneAndInternetBundle` is created as a new analysis feature that combines `PhoneService` and `InternetService` information into a single service classification. This feature is used to compare customer distributions across individual and bundled services, with the figure presenting service populations alongside their corresponding churn rates.

In [ ]:
column_metadata["PhoneAndInternetBundle"] = {
        "category": "bundledServices",
        "column": "PhoneAndInternetBundle",
        "population": {
            "has_service": ["DSL", "Fiber optic", "Yes"],
            "service_types": ["DSL", "Fiber optic", "Phone"]
        },
        "sub_services": [
            "MultipleLines",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
        ],
        "analysis_values": ["Yes", "No"],
        "unit": None,
        "analysis": "categorical"
}

analysis_df["PhoneAndInternetBundle"] = (
    analysis_df["InternetService"].map({"DSL": "Internet", "Fiber optic": "Internet", "No": "No Internet"})
    + analysis_df["PhoneService"].map({"Yes": " Phone", "No": " No Phone"})
)

plot_feature_list(column_metadata, "bundledServices",analysis_df, labels={"Internet Phone": "Phone & Internet", "Internet No Phone": "Internet Only", "No Internet Phone": "Phone Only"})

#### Bundled Service Service-Level Summary
A statistical summary is generated to examine customer distributions and churn outcomes across specific service combinations.

In [ ]:
analysis_df["ServiceType"] = np.select(
    [
        (analysis_df["InternetService"] == "DSL") &
        (analysis_df["PhoneService"] == "No"),

        (analysis_df["InternetService"] == "Fiber optic") &
        (analysis_df["PhoneService"] == "No"),

        (analysis_df["InternetService"] == "DSL") &
        (analysis_df["PhoneService"] == "Yes"),

        (analysis_df["InternetService"] == "Fiber optic") &
        (analysis_df["PhoneService"] == "Yes")
    ],
    [
        "DSL Only",
        "Fiber Only",
        "DSL & Phone",
        "Fiber & Phone"
    ],
    default="Phone Only"
)

pivot = (
    analysis_df
    .groupby("ServiceType")
    .agg(
        Customers=("Churn", "size"),
        Churned=("Churn", lambda x: (x == "Yes").sum()),
        ChurnRate=("Churn", lambda x: (x == "Yes").mean() * 100)
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
table_ax = ax
table_ax.axis("off")

table_data = pivot[["ServiceType", "Customers", "Churned", "ChurnRate"]].copy()

table_data.columns = ["Service Bundle", "Customer Count", "Churn Count", "Churn Rate (%)"]

table_data["Churn Rate (%)"] = table_data["Churn Rate (%)"].round(1)

# Create the table.
table = table_ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center"
)

table_ax.set_title(
    "Service Bundle Combinations with Churn Statistics",
    fontsize=16,
    fontweight="bold",
    y=.85
)

for col in range(4):
    table[(0, col)].set_text_props(fontweight="bold", fontsize="12")

for row in range(5):
    table[(row, 0)].set_text_props(fontweight="bold", fontsize="12")


table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.tight_layout()
plt.show()

#### Bundeld Phone & Internet Service Analysis
The visualization shows that over two-thirds of customers bundle their phone and internet services, with this group experiencing a churn rate of approximately 33%. Customers that only subscribe to phone service represent roughly 20% of the customer population and have a substantially lower churn rate than customers subscribing to internet services.

The statistical summary provides additional detail into the specific service combinations. All customers subscribing to fiber internet also have phone service, with this group having a comparatively high churn rate of nearly 42%. DSL and Phone and Phone Only have similar customer populations; however, Phone Only has a churn rate less than half that of DSL and Phone. DSL Only has a churn rate of approximately 25%, but represents less than 10% of the customer population, making its overall contribution to customer churn less impactful.

#### Individual Service Analysis
The customer populations subscribing to each individual service are examined to evaluate the adoption of their associated sub-services. These populations are used to compare sub-service distributions and their corresponding churn rates within each individual service category.

##### Data Visualization Functions
The following functions generates visualizations structuring the service analysis into three levels, allowing both customer population and churn behavior to be evaluated at the primary service and sub-service levels.

1. **Primary Service**: Displays the customer population that subscribes to the primary service and its corresponding churn rate.

2. **Sub-Service Adoption**: Displays the proportion of primary service subscribers who also subscribe to each available sub-service.

3. **Sub-Service Churn**: Displays the churn rate for customers within each sub-service group.

In [ ]:
def analyze_service_population(df, config, service_value):
    column = config["column"]
    # Primary Service 
    level_1 = (
        df[column]
        .value_counts(normalize=True)
        .mul(100)
        .rename("PopulationPercent")
        .reset_index()
    )

    level_1.columns = [
        "ServiceType",
        "PopulationPercent"
    ]

    level_1["Service"] = service_value

    population = df.loc[
        df[column] == service_value
    ].copy()
    
    # Sub-Service Adoption
    level_2 = []
    for service in config["sub_services"]:
        counts = (
            population[service]
            .value_counts(normalize=True)
            .mul(100)
        )
        
        for value, percent in counts.items():
            level_2.append({
                "Service": service_value,
                "SubService": service,
                "Value": value,
                "PopulationPercent": percent
            })

    level_2 = pd.DataFrame(level_2)

    # Sub-Service Churn
    level_3 = []

    for service in config["sub_services"]:
        crosstab = (
            pd.crosstab(
                population[service],
                population["Churn"],
                normalize="index"
            )
            .mul(100)
            .reset_index()
        )
        
        for _, row in crosstab.iterrows():
            level_3.append({
                "Service": service_value,
                "SubService": service,
                "Value": row[service],
                "ChurnNo": row.get("No", 0),
                "ChurnYes": row.get("Yes", 0)
            })

    level_3 = pd.DataFrame(level_3)

    return {
        "service": column,
        "service_type": service_value,
        "overall": level_1,
        "subservice_population": level_2,
        "subservice_churn": level_3
    }
    
def add_bar_labels(ax, threshold=3):
    for container in ax.containers:
        labels = []
        for bar in container:
            height = bar.get_height()
            if height >= threshold:
                labels.append(f"{height:.0f}%")
            else:
                labels.append("")

        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            color="white",
            fontsize=8,
            fontweight="bold"
        )
        
def add_stacked_bar_labels(ax, proportions, threshold=3):
    for i, subservice in enumerate(proportions.index):
        yes = proportions.loc[subservice, "Yes"]
        no = proportions.loc[subservice, "No"]
        
        if yes >= threshold:
            ax.text(
                i,
                yes / 2,
                f"{yes:.0f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=8,
                fontweight="bold"
            )

        if no >= threshold:
            ax.text(
                i,
                yes + (no / 2),
                f"{no:.0f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=8,
                fontweight="bold"
            )

In [ ]:
def plot_service_analysis(df, results_1, results_2):
    fig = plt.figure(figsize=(14,14), constrained_layout=True)
    gs = GridSpec(3, 2, figure=fig, height_ratios=[2, 1, 1])
    
    # Level 1: Service Population Visualization
    ax1 = fig.add_subplot(gs[0, :])    
    pie_plot(df, results_1["service"], ax1)
    fig.suptitle(
        f"{format_column_name(results_1["service"])} Distribution",
        fontsize=22,
        fontweight="bold",
        x=0.45,
        y=0.98
    )

    # Level 2: Sub-Service Population Visualization
    ax2 = fig.add_subplot(gs[1, 0])
    population_1 = (
        results_1["subservice_population"]
        .pivot(
            index="SubService",
            columns="Value",
            values="PopulationPercent"
        )
        .fillna(0)
    )

    x = range(len(population_1))

    ax2.bar(
        x,
        population_1["Yes"],
        label="Yes",
        color="#007bff"
    )
    ax2.bar(
        x,
        population_1["No"],
        bottom=population_1["Yes"],
        label="No",
        color="#adb5bd"
    )
    ax2.set_title(f"{results_1['service_type']} — Sub-Service Distributions")
    ax2.set_ylabel("Percentage")
    ax2.set_xlabel("")
    ax2.set_ylim(0, 100)
    ax2.set_xticks(x)
    ax2.set_xticklabels(
        population_1.index,
        rotation=45,
        ha="right"
    )
    
    for spine_name in ("top", "right"):
        ax2.spines[spine_name].set_visible(False)
        
    ax2.legend(
            title="Subscribed",
            loc="upper left",
            bbox_to_anchor=(1, 1),
            reverse=True
        )

    add_stacked_bar_labels(ax2, population_1)
    ax3 = fig.add_subplot(gs[1, 1])
    population_2 = (
        results_2["subservice_population"]
        .pivot(
            index="SubService",
            columns="Value",
            values="PopulationPercent"
        )
        .fillna(0)
    )
    for value in ["Yes", "No"]:
        if value not in population_2.columns:
            population_2[value] = 0

    x = range(len(population_2))
    
    ax3.bar(
        x,
        population_2["Yes"],
        label="Yes",
        color="#007bff"
    )
    ax3.bar(
        x,
        population_2["No"],
        bottom=population_2["Yes"],
        label="No",
        color="#adb5bd"
    )
    ax3.set_title(f"{results_2['service_type']} — Sub-Service Distributions")
    ax3.set_ylabel("Percentage")
    ax3.set_xlabel("")
    ax3.set_ylim(0, 100)
    ax3.set_xticks(x)
    ax3.set_xticklabels(
        population_2.index,
        rotation=45,
        ha="right"
    )
    
    for spine_name in ("top", "right"):
        ax3.spines[spine_name].set_visible(False)
    
    ax3.legend(
            title="Subscribed",
            loc="upper left",
            bbox_to_anchor=(1, 1),
            reverse=True
        )
    add_stacked_bar_labels(ax3, population_2)
        
    # Level 3: Churn Visualization
    ax4 = fig.add_subplot(gs[2,0])
    sns.barplot(
        data=results_1["subservice_churn"],
        x="SubService",
        y="ChurnYes",
        hue="Value",
        ax=ax4,
        palette=churn_palette,
        saturation=1
    )
    ax4.set_title(f"{results_1["service_type"]} Customers — Sub-Service Churn")
    ax4.set_ylabel("Churn Rate (%)")
    ax4.set_xlabel("")
    ax4.set_ylim(0, 100)
    plt.setp(
        ax4.get_xticklabels(),
        rotation=45,
        ha="right"
        )
    
    for spine_name in ("top", "right"):
        ax4.spines[spine_name].set_visible(False)
    
    ax4.legend(
            title="Churn",
            loc="upper left",
            bbox_to_anchor=(1, 1)
        )
    add_bar_labels(ax4)
    
    ax5 = fig.add_subplot(gs[2,1])
    sns.barplot(
        data=results_2["subservice_churn"],
        x="SubService",
        y="ChurnYes",
        hue="Value",
        ax=ax5,
        palette=churn_palette,
        saturation=1
    )
    ax5.set_title(f"{results_2["service_type"]} Customers — Sub-Service Churn")
    ax5.set_ylabel("Churn Rate (%)")
    ax5.set_xlabel("")
    ax5.set_ylim(0, 100)
    plt.setp(
        ax5.get_xticklabels(),
        rotation=45,
        ha="right"
        )
    
    for spine_name in ("top", "right"):
        ax5.spines[spine_name].set_visible(False)
        
    ax5.legend(
            title="Churn",
            loc="upper left",
            bbox_to_anchor=(1, 1)
        )
    add_bar_labels(ax5)

    plt.tight_layout(rect=[0, 0, 0.90, 0.94])
    plt.show()
    
def plot_single_service_analysis(df, results):
    fig = plt.figure(figsize=(7, 14))
    gs = GridSpec(3, 1, figure=fig,  height_ratios=[1.5, 1, 1])

    fig.suptitle(
        f"{format_column_name(results['service'])} Data",
        fontsize=22,
        fontweight="bold",
        x=0.45,
        y=0.98
    )

    # Level 1: Overall service population
    ax1 = fig.add_subplot(gs[0])
    pie_plot(
        df,
        results["service"],
        ax1
    )
    ax1.set_title(
        f"{format_column_name(results['service'])} Distribution",
        fontsize=14,
        fontweight="bold"
    )

    # Level 2: Sub-service population
    ax2 = fig.add_subplot(gs[1])
    population_pivot = (
        results["subservice_population"]
        .pivot(
            index="SubService",
            columns="Value",
            values="PopulationPercent"
        )
        .fillna(0)
    )
    
    x = np.arange(len(population_pivot))

    ax2.bar(
        x,
        population_pivot["Yes"],
        label="Yes",
        color="#007bff"
    )

    ax2.bar(
        x,
        population_pivot["No"],
        bottom=population_pivot["Yes"],
        label="No",
        color="#adb5bd"
    )

    ax2.set_xticks(x)
    ax2.set_xticklabels(population_pivot.index)

    if results["service_type"] == "Yes":
        ax2.set_title(
            f"{format_column_name(results['service'])} — Sub-Service Distribution",
            fontsize=14,
            fontweight="bold"
        )
    else:
        ax2.set_title(
            f"{results['service_type']} — Sub-Service Distribution",
            fontsize=14,
            fontweight="bold"
        )

    ax2.set_ylabel("Percentage")
    ax2.set_xlabel("")
    ax2.set_ylim(0, 100)

    plt.setp(
        ax2.get_xticklabels(),
        ha="center"
    )

    ax2.legend(
        title="Subscribed",
        loc="upper left",
        bbox_to_anchor=(1, 1),
        reverse=True
    )
    for spine_name in ("top", "right"):
        ax2.spines[spine_name].set_visible(False)

    for i, subservice in enumerate(population_pivot.index):
        yes_value = population_pivot.loc[subservice, "Yes"]
        no_value = population_pivot.loc[subservice, "No"]

        if yes_value > 3:
            ax2.text(
                i,
                yes_value / 2,
                f"{yes_value:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=10,
                fontweight="bold"
            )

        if no_value > 3:
            ax2.text(
                i,
                yes_value + (no_value / 2),
                f"{no_value:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=10,
                fontweight="bold"
            )

    # Level 3: Sub-service churn rate
    ax3 = fig.add_subplot(gs[2])
    sns.barplot(
        data=results["subservice_churn"],
        x="SubService",
        y="ChurnYes",
        hue="Value",
        ax=ax3,
        palette=churn_palette,
        saturation=1,
        width=.5
    )

    if results["service_type"] == "Yes":
        ax3.set_title(
            f"{format_column_name(results['service'])} Customers — Sub-Service Churn",
            fontsize=14,
            fontweight="bold"
        )
    else:
        ax3.set_title(
            f"{results['service_type']} Customers — Sub-Service Churn",
            fontsize=14,
            fontweight="bold"
        )

    ax3.set_ylabel("Churn Rate (%)")
    ax3.set_xlabel("")
    ax3.set_ylim(0, 100)

    plt.setp(ax3.get_xticklabels(), ha="center")

    ax3.legend(
        title="Churn",
        loc="upper left",
        bbox_to_anchor=(1, 1)
    )

    add_bar_labels(ax3)
    
    for spine_name in ("top", "right"):
        ax3.spines[spine_name].set_visible(False)

    plt.tight_layout(rect=[0, 0, 0.90, 0.94])
    plt.show()


#### Phone Service Visualization
The population of customers with `PhoneService` is examined to determine the proportion that also subscribe to `MultipleLines` and the corresponding churn rate among these customers.

In [ ]:
config = column_metadata["PhoneService"]

phone_results = analyze_service_population(df, config, "Yes")

plot_single_service_analysis(analysis_df, phone_results)

#### Phone Service Analysis
- Approximately 90% of customers subscribe to `PhoneService`, suggesting that phone service itself may have limited discriminatory value when predicting churn due to the relatively small proportion of customers without it.

- Among customers with phone service, the distribution between those with `MultipleLines` and those without is relatively balanced. Customers with multiple lines have a slightly higher churn rate than those without multiple lines; however, the difference is relatively small, suggesting that having multiple lines alone may not be a particularly strong indicator of churn.

#### Internet Service Visualization
`InternetService` is first visualized to compare the distribution of customers with internet service against those without, while further distinguishing customers with `DSL` and `Fiber optic` service. 

The populations of customer with either `DSL` or `Fiber optic` are then examined separately to compare the distribution of customers that subscribe to `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, and `StreamingMovies`. For each of these service groups, churn rates are evaluated to identify whether specific internet service combinations are associated with higher or lower customer churn.


In [ ]:
config = column_metadata["InternetService"]

fiber_results = analyze_service_population(analysis_df, config, "Fiber optic")

dsl_results = analyze_service_population(analysis_df, config, "DSL")

plot_service_analysis(analysis_df, fiber_results, dsl_results)

#### Internet Service Analysis
- `InternetService` accounts for approximately 80% of customers population. Among internet customers, `Fiber optic` accounts for more than half of subscriptions.

- The population of internet subscribes with `OnlineSecurity` and `TechSupport` is ignificantly lower for `Fiber optic` versus `DSL`; while `Fiber optic` customers have substantially higher subscription rates for `StreamingTV` and `StreamingMovies`.

- Among customers subscribed to each internet subservice, `Fiber optic` customers consistently exhibit higher churn rates than `DSL` customers. This suggests that Fiber internet service may have a stronger discriminatory value for predicting churn, particularly when considered alongside the customer's subscribed internet services.

#### Fiber Internet Service Visualization & Statistics
The population of fiber customers is visualized separately to provide a clearer understanding of churn rates across individual fiber internet subservices. 

In addition, a statistical report is generated to further analyze the relationship between each subservice’s churn rate and the overall churn rate among fiber customers. This analysis helps identify which subservices may have the greatest contribution to customer churn within the fiber service population.

In [ ]:
analysis_df["FiberSubService"] = np.select([
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["OnlineSecurity"] == "Yes"),
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["OnlineBackup"] == "Yes"),
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["DeviceProtection"] == "Yes"),
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["TechSupport"] == "Yes"),
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["StreamingTV"] == "Yes"),
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["StreamingMovies"] == "Yes"),
],
[
    "Fiber & Online Security",
    "Fiber & Online Backup",
    "Fiber & Device Protection",
    "Fiber & TechSupport",
    "Fiber & Streaming TV",
    "Fiber & Streaming Movies"
], 
    default="Fiber Only")

pivot = (
    analysis_df
    .groupby("FiberSubService")
    .agg(
        Customers=("Churn", "size"),
        Churned=("Churn", lambda x: (x == "Yes").sum()),
        ChurnRate=("Churn", lambda x: (x == "Yes").mean() * 100)
    )
    .reset_index()
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(13, 8),
    gridspec_kw={"height_ratios": [1,1.5]}
)

chart_ax = axes[0]

sns.countplot(data=analysis_df, x="FiberSubService", hue="Churn", palette=churn_palette, saturation=1, ax=chart_ax)


chart_ax.set_title(
    "Fiber Optic Subservice Adoption with Churn Status",
    fontsize=16,
    fontweight="bold"
)
chart_ax.set_xlabel("")
chart_ax.set_ylabel("Customers",fontweight="bold")

for spine_name in ("top", "right"):
    chart_ax.spines[spine_name].set_visible(False)

chart_ax.legend(
    title="Churn",
    loc="upper left",
    bbox_to_anchor=(1, 1) 
)

plt.setp(
        chart_ax.get_xticklabels(), rotation=45, ha="right")

table_ax = axes[1]
table_ax.axis("off")

table_data = pivot[["FiberSubService", "Customers", "Churned", "ChurnRate"]].copy()

table_data.columns = ["Fiber Package", "Customer Count", "Churn Count", "Churn Rate (%)"]

table_data["Churn Rate (%)"] = table_data["Churn Rate (%)"].round(1)

table = table_ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center"
)

table_ax.set_title(
    "Fiber Optic Package Churn Statistics",
    fontsize=16,
    fontweight="bold",
    y=.85
)

for col in range(4):
    table[(0, col)].set_text_props(fontweight="bold", fontsize="12")

for row in range(8):
    table[(row, 0)].set_text_props(fontweight="bold", fontsize="12")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.tight_layout()
plt.show()


#### Fiber Internet Service Analysis
- The Fiber Only customer population is significantly larger than the populations of customers who subscribe to any individual fiber subservice. However, its churn rate is substantially lower than the churn rates observed among most fiber subservice subscribers.

- `OnlineBackup` and `OnlineSecurity` have the largest subscription populations among the individual fiber subservices.. Despite their relatively large customer populations, their churn rates differ considerably.

- Despite having a smaller customer population than Online Backup and Online Security, `DeviceProtection` has a relatively high churn rate worth further analysing.

- `StreamingMovies` and `StreamingTV` have the highest churn rates among the fiber subservices,suggesting that customers subscribing to these services may represent distinct customer segments.

- `TechSupport` represents the smallest customer population among the fiber subservices; however, despite its small population, its churn rate of 38.18% warrants further analysis.

To further investigate the relationship between fiber subservices and customer churn, additional features are engineered to identify customers who do not subscribe to specific services. 
`FiberNoSecurity`, `FiberNoBackup`, `FiberNoDeviceProtection`, and  `FiberNoTechSupport` are crated as new analysis features to identify customers who do not subscribe to specific services to determine whether the presence or absence of a particular service is associated with increased churn, without assuming that a high churn rate among subscribers indicates that the service itself causes churn.

In [ ]:
analysis_df["FiberNoSecurity"] = np.select(
    [
        analysis_df["InternetService"] != "Fiber optic",
        analysis_df["OnlineSecurity"] == "No"
    ],
    [
        "No fiber subscription",
        "Yes"
    ],
        default="No"
)

analysis_df["FiberNoBackup"] = np.select(
    [
        analysis_df["InternetService"] != "Fiber optic",
        analysis_df["OnlineBackup"] == "No"
    ],
    [
        "No fiber subscription",
        "Yes"
    ],
        default="No"
)

analysis_df["FiberNoDeviceProtection"] = np.select(
    [
        analysis_df["InternetService"] != "Fiber optic",
        analysis_df["DeviceProtection"] == "No"
    ],
    [
        "No fiber subscription",
        "Yes"
    ],
        default="No"
)

analysis_df["FiberNoTechSupport"] = np.select(
    [
        analysis_df["InternetService"] != "Fiber optic",
        analysis_df["TechSupport"] == "No"
    ],
    [
        "No fiber subscription",
        "Yes"
    ],
        default="No"
)


### Investigate Relationships Between Features & Churn Rates
To further investigate potential indicators of customer churn, the features previously identified as potential churn predictors will be compared with features from other customer categories. Examining these relationships will help determine whether certain customer characteristics are associated with higher or lower churn rates and whether patterns observed within individual feature categories are also reflected across other categories.

#### Customer Tenure vs. Contract Agreement Visualization
Customer tenure and contract agreement are visualized to examine how the length of a customer’s relationship with the company and their contractual commitment relate to churn.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

sns.boxplot(
    data=analysis_df,
    x="Contract",
    y="tenure",
    hue="Churn",
    palette=churn_palette,
    saturation=1,
    ax=ax
)

ax.set_title(
    "Customer Tenure vs. Contract Type with Churn Status",
    fontsize=16,
    fontweight="bold"
)

ax.legend(
    title="Churn",
    loc="upper left",
    bbox_to_anchor=(1, 1)
)

ax.set_xlabel("Contract Agreement",fontweight="bold", y=.8)
ax.set_ylabel("Customer Tenure (months)", fontweight="bold")

for spine_name in ("top", "right"):
    ax.spines[spine_name].set_visible(False)

plt.tight_layout()
plt.show()

#### Customer Tenure vs. Contract Agreement Analysis
- The median tenure among customers with `month-to-month` contract agreements who churn is approximately 10 months. However, several outliers indicate that some long-tenured customers, with approximately 50–70 months of tenure, also churn while remaining on a month-to-month contract. This suggests that month-to-month contract agreements may be associated with an increased likelihood of churn.

- Customers with `two year` contract agreements are largely represented by customers with higher tenure and have the lowest levels of churn. A small number of outliers with less than approximately 15 months of tenure are also observed among two year contract customers that did not churn, suggesting that longer-term contracts may be associated with greater customer retention.

Based on these observations, customers with 12 months or less of tenure are classified as new customers. `NewM2M` is created as a new analysis feature to identify customers who have both 12 months or less of tenure and a month-to-month contract, allowing this potentially higher-risk customer segment to be analyzed further.

In [ ]:
analysis_df["NewM2M"] = np.where(
    (analysis_df["Contract"] == "Month-to-month") &
    (analysis_df["tenure"] <= 12),
    "Yes",
    "No"
)

#### Monthly Charges vs. Contract Agreement Visualization
`MonthlyCharges` is visualized alongside `Contract` to examine how customer pricing varies across different contract arrangements and how these differences may relate to churn.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

sns.boxplot(
    data=analysis_df,
    x="Contract",
    y="MonthlyCharges",
    hue="Churn",
    palette=churn_palette,
    saturation=1,
    ax=ax
)

ax.set_title(
    "Monthly Charges vs. Contract Agreement with Churn Status",
    fontsize=16,
    fontweight="bold"
)

ax.legend(
    title="Churn",
    loc="upper left",
    bbox_to_anchor=(1, 1)
)

ax.set_xlabel("Contract Agreement",fontweight="bold", y=.8)
ax.set_ylabel("Monthly Charges", fontweight="bold")

for spine_name in ("top", "right"):
    ax.spines[spine_name].set_visible(False)

plt.tight_layout()
plt.show()

#### Montly Charges vs. Contract Agreement Analysis
- The median monthly charge among customers with `month-to-month` contract agreements that churn is approximately $80.

- The median monthly charge among customers with `One year` and `Two year` contract agreements is approximately $95.

Based on these observations, along with the previous analysis of monthly charges, `M2MHighCharge` is created as a new analysis feature to identify customers with month-to-month contracts whose monthly charges fall within approximately the upper 25% of the observed monthly charge range.

In [ ]:
analysis_df["M2MHighCharge"] = np.where(
    (analysis_df["Contract"] == "Month-to-month") &
    (analysis_df["HighMonthlyCharges"] == "Yes"),
    "Yes",
    "No"
)

#### Ineternet Service Type vs. Monthly Charges Statistics & Visualization
The distribution of `Fiber optic` and `DSL` customers is analyzed in cobination with `MonthlyChargeQuartiles` to examine how bot internet service types are represented at different charge levels. 

In [ ]:
service_distribution = (
    analysis_df
    .groupby("MonthlyChargesQuartile")
    .agg(
        Customers=("InternetService", "size"),
        FiberCustomers=("InternetService", lambda x: (x == "Fiber optic").sum()),
        DSLCustomers=("InternetService", lambda x: (x == "DSL").sum())
    )
    .reset_index()
)

# Calculate the percentage of customers subscribed to Fiber
service_distribution["FiberPercent"] = (
    service_distribution["FiberCustomers"] /
    service_distribution["Customers"] * 100
)

# Calculate the percentage of customers subscribed DSL
service_distribution["DSLPercent"] = (
    service_distribution["DSLCustomers"] /
    service_distribution["Customers"] * 100
)

# Reshape the data for the bar chart
plot_data = service_distribution.melt(
    id_vars="MonthlyChargesQuartile",
    value_vars=["FiberPercent", "DSLPercent"],
    var_name="InternetService",
    value_name="Percentage"
)

plot_data["InternetService"] = plot_data["InternetService"].replace({
    "FiberPercent": "Fiber optic",
    "DSLPercent": "DSL"
})

fig, ax = plt.subplots(
    2,
    1,
    figsize=(13, 8),
    gridspec_kw={"height_ratios": [1, .5]}
)

chart_ax = ax[0]

sns.barplot(
    data=plot_data,
    x="MonthlyChargesQuartile",
    y="Percentage",
    hue="InternetService",
    hue_order=["DSL", "Fiber optic"],
    palette=palette,
    saturation=1,
    ax=chart_ax
)

chart_ax.set_title(
    "Fiber vs. DSL Representation by Monthly Charges Quartiles",
    fontsize=16,
    fontweight="bold"
)

chart_ax.set_xlabel("Monthly Charges Quartiles", fontweight="bold")
chart_ax.set_ylabel("Customers (%)",fontweight="bold")

for spine_name in ("top", "right"):
    chart_ax.spines[spine_name].set_visible(False)

chart_ax.legend(
    title="Internet Service",
    loc="upper left",
    bbox_to_anchor=(1, 1),
)

table_ax = ax[1]
table_ax.axis("off")

table_data = service_distribution[["MonthlyChargesQuartile", "Customers", "DSLCustomers", "DSLPercent", "FiberCustomers", "FiberPercent"]].copy()

table_data.columns = ["Charge Quartile", "Customers", "DSL Customers", "DSL (%)", "Fiber Customers", "Fiber (%)"]

table_data["DSL (%)"] = table_data["DSL (%)"].round(1)
table_data["Fiber (%)"] = table_data["Fiber (%)"].round(1)

table = table_ax.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc="center",
    cellLoc="center"
)

table_ax.set_title(
    "Internet Service Distribution by Monthly Charges Quartile",
    fontsize=16,
    fontweight="bold",
    y=.85
)

for col in range(6):
    table[(0, col)].set_text_props(fontweight="bold")
    
for row in range(5):
    table[(row, 0)].set_text_props(fontweight="bold")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.tight_layout()
plt.show()

#### Ineternet Service Type vs. Monthly Charges Analysis
- `DSL` customers are heavily concentrated in the lower monthly-charge quartiles.

- `Fiber optic` customers are disproportionately concentrated in the higher monthly-charge quartiles.

- The distribution indicates a strong relationship between `InternetService` type and `MonthlyCharges`, with `Fiber optic` customers becoming increasingly represented as monthly charges move into the higher quartiles.

#### Contract Agreement vs. Monlty Charges Statistics for Fiber Customers
To further investigate the relationship between `Contract` and `MonthlyCharges` among `Fiber optic` customers, monthly charge quartiles are calculated specifically within the customer population subscribed to fiber internet to analyze the distribution of `Fiber optic` customers across `Contract` and `MonthlyCharges`.


In [ ]:
fiber_df = analysis_df[analysis_df["InternetService"] == "Fiber optic"].copy()

fiber_df["MonthlyChargesQuartile"] = pd.qcut(
    fiber_df["MonthlyCharges"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

customer_counts = (
    fiber_df
    .groupby(["Contract", "MonthlyChargesQuartile"])
    .size()
    .unstack()
    .reindex(columns=["Q1", "Q2", "Q3", "Q4"])
)

fig, ax = plt.subplots(figsize=(13, 4))

table_ax = ax

table_ax.axis("off")

table_ax.set_title(
    "Fiber Customer Contract Agreements vs. Monthly Charges",
    fontsize=16,
    fontweight="bold",
    y=.85
)

data = customer_counts.reset_index()

# Two header rows.
header_1 = ["", "", "Monthly Charges Quartiles", "", ""]
header_2 = [""] + list(data.columns[1:])

table_data = (
    [header_1, header_2]
    + data.values.tolist()
)

table = table_ax.table(cellText=table_data, loc="center", cellLoc="center")

table[(0, 1)].visible_edges = "BTL"
table[(0, 2)].visible_edges = "BT"
table[(0, 3)].visible_edges = "BT"
table[(0, 4)].visible_edges = "BTR"

table[(0, 2)].set_text_props(fontweight="bold", ha="center")

for col in range(5):
    table[(1, col)].set_text_props(fontweight="bold")

for row in range(2, 5):
    table[(row, 0)].set_text_props(fontweight="bold")

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

plt.show()

#### Contract Agreement vs. Monlty Charges Analysis for Fiber Customers
Overall, the distribution suggests that among Fiber customers, month-to-month contracts are more prevalent at lower monthly charges, while one and two-year contracts become increasingly represented at higher monthly charges.

#### Contract Agreement vs. Monlty Charges Churn Rate with Churn Rate Visualization for Fiber Customers 
The churn rates of Fiber customers are visualized across contract types and monthly charge quartiles to further examine how these two features interact with customer churn.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

custom_cmap = LinearSegmentedColormap.from_list(
    "blue_to_red", ["#0d6efd", "#0dcaf0", "#ffffff", "#d63384", "#dc3545"])

heatmap_data = (
    fiber_df
    .groupby(["Contract", "MonthlyChargesQuartile"])["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .unstack()
)

quartile_order = ["Q1", "Q2", "Q3", "Q4"]

heatmap_data = heatmap_data.reindex(columns=quartile_order)

fig, ax = plt.subplots(figsize=(13, 6))
ax = ax

sns.heatmap(
    heatmap_data,
    annot=True,
    linewidth=.5,
    fmt=".1f",
    annot_kws={"fontweight": "bold", "fontsize": "11"},
    cmap=custom_cmap,
    cbar_kws={"label": "Churn Rate (%)"},
    ax=ax
)

ax.set_title(
    "Fiber Customers: Churn Rate by Contract and Monthly Charges",
    fontsize=16,
    fontweight="bold",
)

plt.xlabel("Monthly Charges Quartiles", fontweight="bold")
plt.ylabel("Contract Agreement",fontweight="bold")
plt.subplots_adjust(top=0.90, right=0.90, hspace=0.35)
plt.show()

#### Contract Agreement vs. Monlty Charges with Churn Rate Analysis for Fiber Customers 
- Fiber customers with `Month-to-month` contract agreements exhibit a higher likelihood of churn compared with customers on longer-term contracts, indicating that `month-to-month` agreements may be an important indicator of churn within the Fiber customer population.

- Churn rates also increase as `MonthlyCharges` increase among Fiber customers with `One year` and `Two year` contracts, indicating that higher monthly charges may be associated with increased churn even among customers with longer-term agreements.

Based on these observations, `FiberM2M` and `FiberM2MHighCharge` are created as analysis features. 
- `FiberM2M` identifies Fiber customers with `Month-to-month` contracts.

-  `FiberM2MHighCharge` identifies Fiber customers with both a `Month-to-month` contract agreement and higher `MonthlyCharges`.

In [ ]:
analysis_df["FiberM2M"] = (
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["Contract"] == "Month-to-month")
)

analysis_df["FiberM2MHighCharge"] = np.where(
    (analysis_df["InternetService"] == "Fiber optic") &
    (analysis_df["Contract"] == "Month-to-month") &
    (analysis_df["HighMonthlyCharges"] == "Yes"),
    "Yes",
    "No"
)

### Feature Assesment
The original features and newly engineered analysis features are assessed to determine their relevance to customer churn.

#### Mutual Information Classification Assesment
A mutual information classification report is generated to measure the degree of dependency between each feature and the churn outcome. This assessment provides a quantitative method for identifying features that contain useful information about churn and determining whether the analysis features engineered provide additional predictive value beyond the original features.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = analysis_df.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
).copy()

y = analysis_df["Churn"].map({"No": 0, "Yes": 1})

categorical_cols = X.select_dtypes(include=["object", "category"]).columns

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns

X[categorical_cols] = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
).fit_transform(X[categorical_cols])

categorical_mask = X.columns.isin(categorical_cols)

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=categorical_mask,
    random_state=42
)
mi_report = pd.DataFrame({
    "Feature": X.columns,
    "MutualInformation": mi_scores
})

mi_report = mi_report.sort_values(
    "MutualInformation",
    ascending=False
).reset_index(drop=True)

display(mi_report)

#### Mutual Information Classification Results
The mutual information results indicate that `Contract` has the strongest relationship with churn, followed by the engineered `FiberM2M` feature, the original `tenure`, and the `TenureQuartile` features. Service-related features such as `OnlineSecurity`, `TechSupport`, `FiberNoSecurity`, and `FiberNoTechSupport` also show relatively stronger relationships with churn compared to the remaining features.

At the lower end of the mutual information scores, features such as `gender`, `PhoneService`, `MultipleLines`, `SeniorCitizen`, `Partner`, `Dependents`, `PaperlessBilling`, and `AutoPay` show relatively weak individual relationships with churn. Similarly, features such as `HighMonthlyCharges`, `HighTotalCharges`, and `MonthlyChargesQuartile` provide limited additional information compared with their underlying continuous or categorical features. While these scores indicate weaker individual relationships, mutual information measures each feature independently and does not account for how features may interact with one another within a predictive model.

Because **XGBoost** can identify nonlinear relationships and interactions between features, the original `SeniorCitizen`, `PhoneService`, and `MultipleLines` features will be retained despite their relatively low mutual information scores, as these variables may still provide useful information when combined with other predictors.

Conversely, engineered analysis features such as `TenureQuartile`, `MonthlyChargesQuartile`, `FiberSubService`, and `HighTotalCharges` will be excluded where their information can be represented by the corresponding original features or where they provide limited additional information.

This approach prioritizes the original features while allowing XGBoost to determine useful thresholds, interactions, and nonlinear relationships without unnecessarily duplicating information through manually discretized features.

In [ ]:
drop_cols = ["TenureQuartile", "gender", "MonthlyChargesQuartile", "FiberSubService", "HighTotalCharges"]

analysis_df = analysis_df.drop(columns=drop_cols)

## 3. Modify
---
During the Modify stage, the dataset is prepared for the customer churn prediction model by applying the feature selections and transformations identified during the Explore stage. Features are removed or retained based on their relevance and analytical value, while the remaining variables are transformed into a format suitable for model development.

### Data Cleaning
The task of cleaning the data prepares the dataset for modeling by addressing data quality issues and ensuring that the selected features are properly formatted and consistent. This includes handling missing values, correcting data types, and removing unnecessary or redundant information to produce a reliable dataset for model development.

Since the data cleaning, transformations, and feature engineering were previously performed on `analysis_df` during the Explore stage, analysis_df will now be copied to the orginal DataFrame. his allows the modified dataset to serve as the primary DataFrame for the remaining modeling steps while preserving `analysis_df` as a reference to the completed exploratory analysis.

In [ ]:
df = analysis_df.copy()

#### Removal of Identifier Variables
`customerID` is dropped from the dataset because it serves as a unique identifier rather than a meaningful predictor of customer churn.

In [ ]:
df = analysis_df.drop(columns=["customerID"])

#### Handel Missing Values

In [ ]:
df.dropna(inplace=True)

### Define Independent & Dependent Variables
The dataset is separated into independent and dependent variables to prepare the data for model development. The independent variables `X` contain the customer features used to predict churn, while the dependent variable `y` represents the `Churn`. The churn values are converted from categorical labels to binary values, where `1` represents customers who churned and `0` represents customers who did not churn.

In [ ]:
# Independent variable
X = df.drop(columns="Churn")

# Dependenent variable
y = df["Churn"].map({"Yes": 1, "No": 0})

### Partition Data into Training & Testing Sets
The dataset is partitioned into training and testing sets to evaluate the model's ability to generalize to unseen customer data. 80% of the data is used to train the model, while the remaining 20% is reserved for testing. This separation allows model performance to be evaluated on data that was not used during training.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### One-Hot Encoding
Categorical features are transformed into numerical representations using one-hot encoding so they can be processed by the XGBoost model. The categorical and numerical features are first identified from the training data, with `OneHotEncoder` converting each categorical value into a separate binary feature.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns

numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            # Ensure that previously unseen categories in the testing data do not cause errors during transformation
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    # Numerical features are passed through unchanged
    remainder="passthrough"
)

#### Feature Selection
Feature selection was previously performed during the Explore stage using mutual information analysis and exploratory findings. Since the modified dataset was created by copying `analysis_df`, the identified feature selections and engineered features have already been incorporated into `df`. Therefore, no additional feature selection is required at this stage, and the existing feature set will be carried forward into model development.

## 4. Model
---
The Model stage uses the prepared dataset to develop and evaluate a customer churn prediction model. The selected features are provided to an XGBoost classifier to identify patterns associated with customer churn and generate predictions. 

### XGBoost
XGBoost is used to develop the customer churn prediction model. XGBoost builds an ensemble of decision trees sequentially, with each tree learning from the errors of previous trees. This approach allows the model to capture nonlinear relationships and interactions between customer features that may contribute to churn.

In [ ]:
xgb_model = XGBClassifier(
    # Number of boosting trees
    n_estimators=200,
    # Maximum depth of each decision tree
    max_depth=4,
    # Step size applied during boosting
    learning_rate=0.05,
    # Ensures reproducible results
    random_state=42
)

The preprocessor and XGBoost model are then combined into a single pipeline. This ensures that categorical features are encoded before being passed to the model and that the same transformations are consistently applied to both the training and testing data

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

### K-Fold Cross Validation
Five-fold cross-validation is used to evaluate the consistency and generalizability of the XGBoost model during training. The training data is divided into five folds, with the model trained on four folds and validated on the remaining fold. This process is repeated five times, allowing each fold to serve as the validation set once. The resulting performance scores are then compared to provide a more reliable estimate of how the model performs across different subsets of the training data.

In [ ]:
cv = StratifiedKFold(
    # Split the training data into 5 folds
    n_splits=5,
    # Randomly shuffle data before creating folds
    shuffle=True,
    # Ensures reproducible fold assignments
    random_state=42
)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    # Use the 5-fold stratified cross-validation
    cv=cv,
    # Evaluate each fold using ROC-AUC             
    scoring="roc_auc"     
)
# Display ROC-AUC for each fold and the average ROC-AUC
print("CV scores:", scores) 
print("Mean ROC-AUC:", scores.mean())


### K-Fold Cross Validation Results
The five-fold cross-validation produced ROC-AUC scores ranging from 0.84 to 0.85, with a mean ROC-AUC of 0.84. The relatively narrow range across the five folds indicates that the XGBoost model performs consistently across different subsets of the training data. The mean ROC-AUC of 0.84 also indicates that the model demonstrates strong ability to distinguish between customers who churn and those who do not during cross-validation.

### Model Tuning
Model tuning is performed to identify a combination of XGBoost hyperparameters that improves predictive performance while maintaining generalizability. Different parameter configurations are evaluated using cross-validation, with ROC-AUC used as the primary performance metric for comparing model configurations.

In [ ]:
# Define the hyperparameter values to evaluate
param_grid = {
    # Number of boosting trees
    "model__n_estimators": [100, 200, 300],
    # Maximum depth of each tree. 
    "model__max_depth": [3, 4, 5],
    # Learning rate for each boosting step           
    "model__learning_rate": [0.05, 0.1]       
}

# Evaluate each parameter combination using 5-fold cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    # Use 5-fold cross-validation
    cv=5,
    # Use ROC-AUC to evaluate model performance
    scoring="roc_auc",
    n_jobs=-1
)

# Fit each parameter combination to the training data
grid_search.fit(X_train, y_train)


#### Model Tuning Results
The `GridSearchCV` identified the following XGBoost configuration as the best-performing combination: a learning rate of 0.05, a maximum tree depth of 4, and 100 estimators.

This configuration achieved a mean cross-validation ROC-AUC of 0.84 across the five folds.

The result is consistent with the initial cross-validation performance of 0.84, indicating that the tuning process produced only a modest improvement in overall model performance.

### Final Model Selection
The best-performing XGBoost configuration identified through hyperparameter tuning is selected as the final model. This model is trained using the entire testing dataset.

In [ ]:
# Select the best-performing model identified by GridSearchCV as the final model
final_model = grid_search.best_estimator_
# Train final model on the full training dataset
final_model.fit(X_train, y_train)

## 5. Assess
---
During the Assess stage, the XGBoost model selected as the final model is evaluated using the held-out testing dataset to determine how well it generalizes to previously unseen customer data. Multiple performance metrics are used to assess the model's ability to distinguish between customers who churn and those who do not, providing a comprehensive evaluation of its predictive performance.

### Model Testing
The final model is evaluated on the held-out testing dataset to examine how its predictions perform on previously unseen customers. Rather than relying on the default classification threshold of 0.50, multiple probability thresholds from 0.20 to 0.60 are tested. This allows the trade-off between precision, recall, F1-score, and accuracy to be examined and helps identify a threshold that provides an appropriate balance for predicting customer churn.

In [ ]:
# Generate the probability that each test customer will churn
y_probability = final_model.predict_proba(X_test)[:, 1]

# Define probability thresholds to evaluate
thresholds = np.arange(0.20, 0.61, 0.05)
results = []

# Evaluate model performance at each threshold
for threshold in thresholds:
    # Classify customers as churned when their predicted probability meets or exceeds the current threshold
    y_pred_threshold = (y_probability >= threshold).astype(int)
    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, y_pred_threshold),
        "Recall": recall_score(y_test, y_pred_threshold),
        "F1": f1_score(y_test, y_pred_threshold),
        "Accuracy": accuracy_score(y_test, y_pred_threshold)
    })
    
# Store the threshold performance results in a DataFrame
threshold_results = pd.DataFrame(results)

# Display performance across all evaluated thresholds
display(threshold_results)

### Model Testing Results
The threshold analysis shows a clear trade-off between precision and recall. As the classification threshold increases, precision generally improves, while recall decreases. At the lowest threshold of 0.20, the model achieves the highest recall of 85.3%, but precision is lower at 45.9%. Conversely, at the highest threshold of 0.60, precision increases to 70.5%, but recall decreases substantially to 36.4%. This demonstrates that a lower threshold allows the model to identify more potential churners at the cost of generating more false-positive predictions.

For this model, a threshold of 0.35 is selected because it provides a strong balance between identifying potential churners and limiting false-positive predictions. At this threshold, the model achieves 71.7% recall, 56.2% precision, an F1-score of 63.0%, and 77.6% accuracy. The 71.7% recall indicates that the model correctly identifies approximately 72% of customers who actually churn, reducing the number of potential churners that are missed. While a lower threshold would capture even more churners, the 0.35 threshold provides a more balanced trade-off between recall and precision. Therefore, customers with a predicted churn probability of 35% or greater will be classified as potential churners.

In [ ]:
churn_threshold = 0.35
# Classify customers as churned when their predicted probability is 35% or greater; otherwise, classify them as not churned
y_pred = (y_probability >= churn_threshold).astype(int)

### Model Performance Evaluation
The final XGBoost model is evaluated using multiple performance metrics.

#### Accuracy
Accuracy measures the overall proportion of customer churn predictions that the model classified correctly. It accounts for both customers who churn and customers who do not churn.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.1%}")

#### Accuracy Results
Nearly 78% of all customer churn predictions are classified correctly. However, accuracy should be interpreted cautiously because the dataset contains a class imbalance between customers who churn and those who do not, the model can achieve relatively high accuracy by correctly predicting the majority class while performing less effectively on the minority churn class. Therefore, accuracy alone does not fully represent the model's ability to identify customers who are at risk of churn

#### Precision
Precision measures the proportion of customers predicted to churn who actually churned. This metric indicates how reliable the model's positive churn predictions are and helps assess the number of false-positive churn predictions produced by the model.

In [ ]:
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.1%}")

#### Precision Results
Approximately 56% of customers predicted to churn actually churned. The remaining predictions represent false positives, where customers were identified as potential churners but did not ultimately churn. This moderate precision is consistent with the model's lower classification threshold, which prioritizes identifying a larger proportion of actual churners.

#### Recall
Recall measures the proportion of customers who actually churned that the model correctly identifies as potential churners. This metric is particularly important for churn prediction because it indicates how effectively the model captures customers who are at risk of leaving while minimizing false negatives.

In [ ]:
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.1%}")

#### Recall Results
The model correctly identified Nearly 72% of customers who actually churned, indicating that the model is effective at capturing a majority of potential churners, which is especially valuable when the goal is to identify customers who may be at risk of leaving. 

#### F1 Score
The F1 score combines precision and recall into a single metric by calculating their harmonic mean. This provides a balanced measure of the model's ability to correctly identify churners while limiting false-positive predictions, making it useful when both precision and recall are important.

In [ ]:
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.1%}")

#### F1 Score Results
The F1 score indicates that the model maintains a reasonable balance between correctly identifying churners and limiting false-positive predictions.

#### Confusion Matrix
A confusion matrix is used to examine the model's classification results in greater detail by comparing the predicted churn classifications with the customers' actual churn outcomes.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Churn", "Churn"],
    cmap=custom_cmap
)

plt.title("XGBoost Churn Prediction")
plt.show()

#### Confusion Matrix Explained

- 824 customers were correctly classified as non-churners (true negatives).

- 209 customers were incorrectly classified as churners who did not actually churn (false positives).

- 106 customers were incorrectly classified as non-churners who ultimately churned (false negatives).

- 268 customers were correctly classified as churners (true positives). 

These results are consistent with the model's emphasis on recall. The model identifies 268 of the 374 actual churners, capturing approximately 71.7% of customers who churned, while 106 churners are missed. Although the model produces some false-positive predictions, the selected threshold allows it to identify a larger proportion of actual churners, which aligns with the objective of identifying customers who may be at risk of leaving.

#### ROC-AUC
ROC-AUC measures the model's ability to distinguish between customers who churn and those who do not across different classification thresholds. The ROC curve compares the true positive rate against the false positive rate at various thresholds, while the AUC (Area Under the Curve) summarizes this performance into a single value. 

In [ ]:
RocCurveDisplay.from_estimator(final_model, X_test, y_test)
print(f"ROC-AUC:   {roc_auc_score(y_test, y_probability):.2f}")

#### ROC-AUC Results
The final model achieves a ROC-AUC of 0.84, indicating a strong ability to distinguish between customers who churn and those who do not across different classification thresholds. This result is also consistent with the 0.847 mean ROC-AUC observed during five-fold cross-validation, suggesting that the model's discriminatory performance is relatively stable when applied to unseen data.

### Feature Importance & Model Interpretability
Individual features are examined to determine their indiviual influence on the model's predictions using **SHAP** (SHapley Additive exPlanations). SHAP shows both the features that are most influential and the direction of their impact on the model's output. Visualized below are the key factors driving churn predictions.


In [ ]:
import shap

preprocessor = final_model.named_steps["preprocessor"]
model = final_model.named_steps["model"]

# Transform the test data
X_test_transformed = preprocessor.transform(X_test)

# Get the names of the transformed features
feature_names = preprocessor.get_feature_names_out()

# Create SHAP explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values.
shap_values = explainer.shap_values(X_test_transformed)
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names)

#### SHAP Results
`Contract`, `tenure`, and `MonthlyCharges` are the most influential predictors, with `Month-to-month contracts` and shorter tenure generally increasing the model to predict a customer's likelihood to churn. while longer-term contracts and greater tenure reduced it. Higher monthly charges were also associated with higher predicted churn.

Other notable factors included lack of `online security`, lack of `technical support`, `electronic-check` payment methods, and `FiberNoSecurity`, which generally contributed to higher churn predictions. 

Features such as two-year contracts and longer tenure tended to reduce predicted churn.

## Conclusion
---
Overall, this XGBoost churn prediction model demonstrates solid predictive performance, achieving 78% accuracy and an 84% ROC-AUC, indicating that the model was generally effective at distinguishing between customers who churn and those who do not.

The SHAP results indicate that the model's churn predictions are driven primarily by customer commitment, length of service, and monthly cost, with additional influence from service options and payment methods.

Together, the model's performance metrics and SHAP analysis provide both a measure of how effectively the model predicts churn and insight into the factors contributing to those predictions, making the results useful for understanding and addressing customer churn.